# End-to-End Case Study - Marketing Response Modeling

This notebook is a closing case study for the course. Its purpose is not to teach a single model family, but to show a full **tabular classification workflow**: early train/test split, feature engineering, leakage-safe preprocessing, baseline comparison with nested cross-validation, focused hyperparameter tuning with Optuna, and final evaluation with both statistical and business-oriented metrics.

The winning model in this case study is a **CatBoost classifier**, but the main lesson is methodological: how to structure model selection and evaluation correctly.


## Requirements

This notebook assumes the following packages are available in the environment:

- `catboost`
- `optuna`
- `imbalanced-learn`

If any of them is missing, install it before running the notebook.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from catboost import CatBoostClassifier
from imblearn.combine import SMOTETomek
from imblearn.pipeline import Pipeline
from optuna import create_study
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.svm import SVC

plt.style.use("seaborn-v0_8-whitegrid")
SEED = 42


This setup cell imports the pieces used throughout the case study. The workflow intentionally stays close to the standard `scikit-learn` ecosystem so that the notebook can be read as a general template for supervised tabular modeling.


## Load the Data and Define the Prediction Task

The target is `Response`: whether the customer accepted the marketing campaign offer.


In [ ]:
data_path = "data/marketing_campaign.csv"
dataset = pd.read_csv(data_path, sep=";")
dataset.head()


This first table is only for orientation. The dataset mixes demographic, spending, and campaign-history variables, which makes it a good example of real tabular classification with both preprocessing and modeling decisions to make.


In [ ]:
target_col = "Response"

print(dataset.info())
display(dataset[target_col].value_counts().rename("count").to_frame())
display((dataset[target_col].value_counts(normalize=True).rename("fraction") * 100).round(2).to_frame())


This quick audit answers the first practical questions: which columns are present, where missing values appear, and how imbalanced the target is. The class balance matters because the business problem is asymmetric: positive responses are the minority but they are the outcome of interest.


## Split Early to Protect the Final Evaluation

The train/test split is performed before any feature engineering or preprocessing choice is fitted. This is the first important anti-leakage decision in the notebook.


In [ ]:
X_raw = dataset.drop(columns=[target_col])
y = dataset[target_col]

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw,
    y,
    test_size=0.20,
    stratify=y,
    random_state=SEED,
)

print(f"train shape: {X_train_raw.shape}")
print(f"test shape:  {X_test_raw.shape}")


Everything that follows should be read as a modeling workflow built on the training partition only. The test set is reserved for a single final evaluation after model selection is complete.


## Feature Engineering as a Reusable Transformation

Instead of mutating the full dataset manually, the notebook packages feature engineering into a reusable function so it can be applied consistently inside the pipeline workflow.


In [ ]:
REFERENCE_DATE = pd.Timestamp("2014-10-04")


def add_features(df):
    X = df.copy()
    X["Dt_Customer"] = pd.to_datetime(X["Dt_Customer"], errors="coerce")

    X["Age"] = REFERENCE_DATE.year - X["Year_Birth"]
    X["SeniorityMonths"] = (REFERENCE_DATE - X["Dt_Customer"]).dt.days / 30.0
    X["Children"] = X["Kidhome"] + X["Teenhome"]
    X["TotalAcceptedCmp"] = X[["AcceptedCmp1", "AcceptedCmp2", "AcceptedCmp3", "AcceptedCmp4", "AcceptedCmp5"]].sum(axis=1)
    X["TotalPurchases"] = X[["NumWebPurchases", "NumCatalogPurchases", "NumStorePurchases"]].sum(axis=1)
    X["TotalMnt"] = X[["MntWines", "MntFruits", "MntMeatProducts", "MntFishProducts", "MntSweetProducts", "MntGoldProds"]].sum(axis=1)

    X = X.drop(columns=["ID", "Dt_Customer", "Z_CostContact", "Z_Revenue", "Year_Birth"], errors="ignore")
    return X


X_train_fe = add_features(X_train_raw)
X_train_fe.head()


The engineered variables are intentionally simple and interpretable: age, customer seniority, total accepted past campaigns, total purchases, total spending, and total number of children at home. The goal is not exhaustive feature invention, but a compact example of domain-informed engineering.


In [ ]:
categorical_features = X_train_fe.select_dtypes(include="object").columns.tolist()
numeric_features = [c for c in X_train_fe.columns if c not in categorical_features]

print("categorical features:", categorical_features)
print("number of numeric features:", len(numeric_features))


This cell fixes the feature groups that will be handled differently in the preprocessing stage. The feature lists are derived from the engineered training data, not from the full dataset.


## Build a Leakage-Safe Preprocessing Pipeline

The preprocessing logic is shared across models: feature engineering, imputation, scaling of numeric variables, and one-hot encoding of categorical variables.


In [ ]:
feature_engineering = FunctionTransformer(add_features, validate=False)

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse=False)),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features),
])


def make_modeling_pipeline(model):
    return Pipeline([
        ("features", feature_engineering),
        ("preprocess", preprocessor),
        ("resample", SMOTETomek(random_state=SEED)),
        ("model", model),
    ])


The pipeline is the structural core of the notebook. Because feature engineering, preprocessing, and resampling all happen inside the same pipeline object, cross-validation can evaluate them without leaking information across folds.


## Baseline Model Comparison with Nested Cross-Validation

The first modeling stage compares a small set of candidate algorithms. The evaluation metric is the positive-class `F1` score, because the business problem is focused on identifying responders rather than maximizing overall accuracy on an imbalanced target.


In [ ]:
search_spaces = [
    {
        "name": "Logistic Regression",
        "pipeline": make_modeling_pipeline(LogisticRegression(max_iter=5000, random_state=SEED)),
        "params": {
            "model__C": [0.1, 1.0, 10.0],
            "model__class_weight": [None, "balanced"],
        },
    },
    {
        "name": "SVC",
        "pipeline": make_modeling_pipeline(SVC(probability=True, random_state=SEED)),
        "params": {
            "model__C": [0.5, 1.0, 5.0],
            "model__gamma": ["scale", 0.01],
            "model__class_weight": [None, "balanced"],
        },
    },
    {
        "name": "Random Forest",
        "pipeline": make_modeling_pipeline(RandomForestClassifier(random_state=SEED)),
        "params": {
            "model__n_estimators": [200],
            "model__max_depth": [None, 10],
            "model__min_samples_leaf": [1, 5],
            "model__class_weight": [None, "balanced"],
        },
    },
    {
        "name": "CatBoost",
        "pipeline": make_modeling_pipeline(
            CatBoostClassifier(
                verbose=0,
                allow_writing_files=False,
                random_state=SEED,
            )
        ),
        "params": {
            "model__iterations": [200],
            "model__depth": [4, 6],
            "model__learning_rate": [0.03, 0.1],
            "model__l2_leaf_reg": [3, 5],
            "model__auto_class_weights": [None, "Balanced"],
        },
    },
]


In [ ]:
outer_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

nested_rows = []
for candidate in search_spaces:
    outer_scores = []
    for train_idx, valid_idx in outer_cv.split(X_train_raw, y_train):
        X_tr = X_train_raw.iloc[train_idx]
        X_val = X_train_raw.iloc[valid_idx]
        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[valid_idx]

        search = GridSearchCV(
            candidate["pipeline"],
            candidate["params"],
            scoring="f1",
            cv=inner_cv,
            n_jobs=None,
        )
        search.fit(X_tr, y_tr)
        y_val_pred = search.best_estimator_.predict(X_val)
        outer_scores.append(f1_score(y_val, y_val_pred))

    nested_rows.append({
        "model": candidate["name"],
        "nested_cv_mean_f1": np.mean(outer_scores),
        "nested_cv_std": np.std(outer_scores),
    })

nested_results = pd.DataFrame(nested_rows).sort_values("nested_cv_mean_f1", ascending=False)
nested_results.round(4)


This table is the baseline model-selection result. The nested structure matters: the inner loop chooses hyperparameters, while the outer loop estimates how that full selection process generalizes. That makes the reported score much more trustworthy than a single ordinary cross-validation run over the full dataset.


## Focused Hyperparameter Tuning for the Winning Model

After the baseline comparison, we tune only the strongest candidate. In this notebook that candidate is CatBoost, so the search budget is concentrated on that model rather than spread across the entire model family comparison.


In [ ]:
def objective_catboost(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 150, 500, step=50),
        "depth": trial.suggest_int("depth", 3, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 10.0),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "verbose": 0,
        "allow_writing_files": False,
        "random_state": SEED,
        "auto_class_weights": "Balanced",
    }

    model = CatBoostClassifier(**params)
    pipe = make_modeling_pipeline(model)
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    scores = cross_val_score(pipe, X_train_raw, y_train, cv=cv, scoring="f1")
    return scores.mean()


In [ ]:
study = create_study(direction="maximize")
study.optimize(objective_catboost, n_trials=40, show_progress_bar=False)

print("Best CV F1:", round(study.best_value, 4))
print("Best parameters:")
study.best_params


The Optuna study now operates on a single well-defined task: improving the training-side cross-validated `F1` of the CatBoost pipeline. This keeps the workflow conceptually clean: first choose the model family, then tune it more deeply.


In [ ]:
trials_df = study.trials_dataframe().sort_values("value", ascending=False)
trials_df[["number", "value", "params_depth", "params_learning_rate", "params_iterations", "params_l2_leaf_reg", "params_subsample"]].head(10)


Inspecting the top trials is often more informative than looking only at the single best parameter set. It gives a rough sense of which regions of the search space are competitive and whether performance is stable or highly sensitive to the tuning choices.


## Fit the Final Model and Evaluate on the Held-Out Test Set

At this point, all model-selection work is finished. The test set is used exactly once, to assess the final tuned pipeline.


In [ ]:
best_catboost = CatBoostClassifier(
    **study.best_params,
    verbose=0,
    allow_writing_files=False,
    random_state=SEED,
    auto_class_weights="Balanced",
)

final_pipe = make_modeling_pipeline(best_catboost)
final_pipe.fit(X_train_raw, y_train)

y_test_pred = final_pipe.predict(X_test_raw)
y_test_score = final_pipe.predict_proba(X_test_raw)[:, 1]

summary_metrics = pd.DataFrame([
    {"metric": "F1 (positive class)", "value": f1_score(y_test, y_test_pred)},
    {"metric": "ROC AUC", "value": roc_auc_score(y_test, y_test_score)},
    {"metric": "Average Precision", "value": average_precision_score(y_test, y_test_score)},
])
summary_metrics.round(4)


In [ ]:
print(classification_report(y_test, y_test_pred, digits=4))


This is the statistical evaluation layer. `F1` focuses on the minority positive class, `ROC AUC` summarizes ranking quality, and `Average Precision` is especially informative for imbalanced classification because it emphasizes precision-recall behavior.


In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(y_test, y_test_pred, ax=ax, colorbar=False)
ax.set_title("Final model on held-out test set")
plt.tight_layout()
plt.show()


The confusion matrix adds the error-type perspective that scalar metrics cannot provide. In a marketing-response task, the relative balance between false positives and false negatives matters because it translates directly into wasted targeting effort versus missed opportunities.


## Business-Oriented Ranking Evaluation: Cumulative Gain and Lift

Because the model will often be used to rank customers by response probability, it is useful to inspect ranking-oriented business metrics in addition to ordinary classification metrics.


In [ ]:
def gains_table(y_true, y_score, n_bins=10):
    df = pd.DataFrame({"y_true": np.asarray(y_true), "y_score": np.asarray(y_score)})
    df = df.sort_values("y_score", ascending=False).reset_index(drop=True)
    df["bucket"] = pd.qcut(np.arange(len(df)), q=n_bins, labels=False)
    df["bucket"] = df["bucket"] + 1

    total_positives = df["y_true"].sum()
    rows = []
    for k in range(1, n_bins + 1):
        top_k = df.iloc[: int(np.ceil(len(df) * k / n_bins))]
        population_fraction = len(top_k) / len(df)
        cumulative_gain = top_k["y_true"].sum() / total_positives
        lift = cumulative_gain / population_fraction
        rows.append({
            "top_fraction": population_fraction,
            "cumulative_gain": cumulative_gain,
            "lift": lift,
        })
    return pd.DataFrame(rows)


gain_df = gains_table(y_test, y_test_score, n_bins=10)
gain_df.round(4)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(gain_df["top_fraction"], gain_df["cumulative_gain"], marker="o", label="model")
axes[0].plot([0, 1], [0, 1], linestyle="--", color="black", label="random baseline")
axes[0].set_xlabel("fraction of customers targeted")
axes[0].set_ylabel("fraction of responders captured")
axes[0].set_title("Cumulative gain curve")
axes[0].legend()

axes[1].plot(gain_df["top_fraction"], gain_df["lift"], marker="o")
axes[1].axhline(1.0, linestyle="--", color="black")
axes[1].set_xlabel("fraction of customers targeted")
axes[1].set_ylabel("lift over random targeting")
axes[1].set_title("Lift curve")

plt.tight_layout()
plt.show()


These curves translate the classifier into a ranking tool. The cumulative-gain curve answers “how many actual responders do we capture if we only contact the top fraction of customers?”, while the lift curve answers “how much better is this targeting strategy than random selection?”


In [ ]:
top_20 = gain_df[gain_df["top_fraction"] == gain_df["top_fraction"].iloc[1]].iloc[0]
pd.DataFrame([
    {
        "targeted_fraction": top_20["top_fraction"],
        "responders_captured": top_20["cumulative_gain"],
        "lift": top_20["lift"],
    }
]).round(4)


This final summary row gives the business-facing reading of the model at a concrete operating point. For example, if the organization can contact only the top 20% of customers, this table summarizes how much of the responder population would be captured and how much better that is than random targeting.


## Closing Remarks

This case study illustrates a complete modeling workflow rather than a single modeling trick. The main methodological lessons are:

- split early and protect the test set,
- place preprocessing and resampling inside the pipeline,
- use nested cross-validation for honest baseline comparison,
- tune only the selected model more deeply,
- and evaluate the final model with both statistical and business-oriented metrics.
